# From DICOM-RT to a research dataset — and back

*A worked example for the [DICOM RT Toolkit](https://github.com/brianmanderson/DicomRtNiftiConverterGUI).*

This notebook takes a public radiotherapy cohort — CT, structure sets **and dose** — from raw
DICOM to an analysis-ready NIfTI dataset, then turns the masks back into a DICOM RT structure
set and measures what the round trip cost.

It follows the same arc as Session 1 of the AAPM 2026 *From Pixels to Patients* track, which
teaches this pipeline in pure Python with
[DicomRTTool](https://github.com/brianmanderson/Dicom_RT_and_Images_to_Mask). Here the work is
done by a compiled cross-platform CLI instead, and the notebook drives it. Three sections have
no counterpart in that session, because the Python tooling cannot do them: **dose export**
(§8-§10), the **reverse direction** (§12), and **conformance against analytic ground truth**
(§13).

**You do not need .NET installed.** Section 0 downloads a self-contained binary.

---
## 0 · Setup

Two things to get: the Python packages this notebook uses for inspection and plotting, and the
toolkit CLI itself.

The CLI is a compiled binary, not a pip package. There are three ways to point at one, tried in
this order:

1. the `RT_CLI` environment variable, if you already have a build you want to use;
2. a local `dotnet run`, if you set `USE_LOCAL_BUILD = True` and have the .NET 8 SDK plus the
   SimpleITK native staged (see **Build instructions** in the repository's `README.md`). This
   path runs `dotnet run --project ../src/DicomRtNifti.Cli`, so it only resolves when the
   notebook's working directory is this `examples/` folder;
3. **the default** — download the self-contained release for your platform, which bundles the
   SimpleITK native and needs no .NET install at all.

> **On the release tag.** `latest-build` is a rolling tag: it is deleted and recreated on every
> push to main. The URL is stable, the bytes are not. Pin a dated tag if you need a run you can
> reproduce byte-for-byte months from now.

In [ ]:
%pip install -q tcia_utils SimpleITK pandas matplotlib numpy pydicom

In [ ]:
import json, os, platform, shutil, subprocess, sys, tarfile, zipfile
from collections import namedtuple
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt

# ---- knobs ---------------------------------------------------------------
N_PATIENTS       = 30                       # of the 40 in the collection
COLLECTION       = "Pancreatic-CT-CBCT-SEG"
OUTPUT_SPACING   = (1.0, 1.0, 3.0)          # mm; None keeps each series' native grid
SALT             = "AAPM2026-PANC"          # fixes the anonymization hashes
CLI_RELEASE_TAG  = "latest-build"
USE_LOCAL_BUILD  = False

# Keep this short on Windows. The DICOM tree nests a 64-character SeriesInstanceUID under the
# work directory, and a long base path pushes the result past the 260-character limit.
WORK_DIR   = Path(os.environ.get("RT_WORK_DIR", "C:/rt_ex" if os.name == "nt" else "~/rt_ex")).expanduser()
DICOM_DIR  = WORK_DIR / "dicom"
NIFTI_DIR  = WORK_DIR / "nifti"
TOOL_DIR   = WORK_DIR / ".tool"

for d in (WORK_DIR, DICOM_DIR, NIFTI_DIR, TOOL_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("working in", WORK_DIR)

In [ ]:
REPO = "brianmanderson/DicomRtNiftiConverterGUI"

def _rid():
    system, machine = platform.system(), platform.machine().lower()
    if system == "Windows":
        return "win-x64", "zip"
    if system == "Linux":
        return "linux-x64", "tar.gz"
    if system == "Darwin":
        return ("osx-arm64", "tar.gz") if machine in ("arm64", "aarch64") else ("osx-x64", "tar.gz")
    raise RuntimeError(f"unsupported platform: {system}/{machine}")


def ensure_cli():
    """Resolve the toolkit CLI, downloading a self-contained build if needed."""
    if os.environ.get("RT_CLI"):
        return [os.environ["RT_CLI"]]

    if USE_LOCAL_BUILD:
        if not shutil.which("dotnet"):
            raise RuntimeError("USE_LOCAL_BUILD is set but 'dotnet' is not on PATH.")
        return ["dotnet", "run", "--project", "../src/DicomRtNifti.Cli", "-c", "Release", "--"]

    rid, ext = _rid()
    exe = TOOL_DIR / ("DicomRtNifti.Cli.exe" if os.name == "nt" else "DicomRtNifti.Cli")
    if exe.exists():
        return [str(exe)]

    asset = f"DicomRtNifti-cli-{rid}.{ext}"
    url = f"https://github.com/{REPO}/releases/download/{CLI_RELEASE_TAG}/{asset}"
    archive = TOOL_DIR / asset
    print(f"downloading {asset} ...")
    urlretrieve(url, archive)

    if ext == "zip":
        with zipfile.ZipFile(archive) as z:
            z.extractall(TOOL_DIR)
    else:
        with tarfile.open(archive) as t:
            t.extractall(TOOL_DIR)

    if not exe.exists():
        found = [p for p in TOOL_DIR.rglob(exe.name)]
        if not found:
            raise RuntimeError(f"{exe.name} not found in {asset}")
        exe = found[0]
    if os.name != "nt":
        exe.chmod(0o755)
    return [str(exe)]


CLI = ensure_cli()
print("CLI:", " ".join(CLI))

In [ ]:
CliResult = namedtuple("CliResult", "returncode stdout stderr")

_EXIT_MEANING = {0: "ok", 1: "conversion failure", 2: "bad arguments"}


def run_cli(*args, check=True, parse_json=False, echo=True):
    """Invoke the toolkit CLI.

    stdout is machine-readable (one JSON document for the cohort modes); stderr carries
    progress. Raising on a non-zero exit is what makes `nbconvert --execute` a real gate on
    this notebook rather than a document that renders whatever happened.
    """
    proc = subprocess.run([*CLI, *map(str, args)], capture_output=True, text=True)
    if echo:
        tail = [ln for ln in proc.stderr.splitlines() if ln.strip()][-3:]
        for line in tail:
            print("   ", line)
    if check and proc.returncode != 0:
        meaning = _EXIT_MEANING.get(proc.returncode, "unknown")
        raise RuntimeError(
            f"CLI exited {proc.returncode} ({meaning})\n"
            + "\n".join(proc.stderr.splitlines()[-20:]))
    if parse_json:
        return json.loads(proc.stdout)
    return CliResult(proc.returncode, proc.stdout, proc.stderr)

One check before anything else. The toolkit's image I/O is SimpleITK, loaded as a native
library, and a native that fails to load is the single most common setup problem. `--version`
constructs a one-voxel image to force the load, so this assert either passes or tells you
exactly what is missing — the same gate the project's CI uses on all three platforms.

In [ ]:
version = run_cli("--version", echo=False)
print(version.stdout.strip())
assert "SimpleITK native: OK" in version.stdout, version.stdout + version.stderr

---
## 1 · Download the cohort

[**Pancreatic-CT-CBCT-SEG**](https://www.cancerimagingarchive.net/collection/pancreatic-ct-cbct-seg/)
— 40 patients treated for pancreatic cancer, each with a planning CT, several cone-beam CTs
aligned to it, structure sets, and a dose distribution.
DOI [10.7937/TCIA.ESHQ-4D90](https://doi.org/10.7937/TCIA.ESHQ-4D90), released **CC BY 4.0**.

Two reasons this collection rather than a lung one:

- **Every patient has dose.** Most public segmentation cohorts ship images and contours only,
  which makes it impossible to demonstrate the dose half of the pipeline.
- **Every patient is a genuinely messy study** — five image series all reporting `Modality=CT`,
  three structure sets, one dose, all under a single study. Choosing correctly among them is a
  real step, and §3 is about doing it deliberately rather than by accident.

> **Budget for the download.** 30 patients × ~5 series each is on the order of tens of GB and
> can take well over an hour. Drop `N_PATIENTS` to 2 or 3 for a first pass — every cell below
> behaves identically, just faster.

> **"SEG" in the name is not DICOM SEG.** The segmentations here are RT structure sets, which is
> what this toolkit reads. See
> [`DicomSegOverview.md`](https://github.com/brianmanderson/Dicom_RT_Images_Csharp/blob/main/DicomSegOverview.md)
> in the research repository for why the two formats coexist and when you would want each.

In [ ]:
from tcia_utils import nbia

series = nbia.getSeries(collection=COLLECTION, format="df")
print(f"{len(series)} series, {series.PatientID.nunique()} patients")
print(series.Modality.value_counts().to_dict())

# Take the first N patients that carry all three of image, structures and dose. Every patient
# in this collection does — but assert it rather than assume it, so the same cell still behaves
# on a collection where that is not true.
have = (series.groupby("PatientID").Modality
        .agg(lambda m: {"CT", "RTSTRUCT", "RTDOSE"}.issubset(set(m))))
eligible = sorted(have[have].index)
assert len(eligible) >= N_PATIENTS, f"only {len(eligible)} complete patients available"

patients = eligible[:N_PATIENTS]
subset = series[series.PatientID.isin(patients)]
print(f"\nselected {len(patients)} patients / {len(subset)} series")

In [ ]:
# Downloads to DICOM_DIR/<SeriesInstanceUID>/*.dcm. Resumable: already-present series are
# skipped, so re-running this cell after an interruption is cheap.
nbia.downloadSeries(subset, input_type="df", path=str(DICOM_DIR))

print("series folders on disk:", len(list(DICOM_DIR.iterdir())))

---
## 2 · Discover what is actually there

`--cohort-scan` walks the tree, groups files into a patient → study → series hierarchy, and
links each structure set and dose to the image series it belongs to. It writes a single JSON
document to stdout and converts nothing.

The interesting field is `match_rule`, which records *how confident* each link is:

| rule | meaning |
|---|---|
| `ReferencedSeriesUid` | the RT object names its image series explicitly. Authoritative. |
| `FrameOfReferenceUid` | matched on the shared frame of reference, resolving ties toward the fullest series. Correct in most studies, a guess when several series share a frame — exactly this collection's situation. |
| `LargestSeriesFallback` | nothing matched; the largest image series was assumed. Look at these. |

A cohort where everything resolved by fallback is not a cohort you should train on yet.

In [ ]:
scan = run_cli("--cohort-scan", "--input", DICOM_DIR, parse_json=True)

assert scan["unreadable_files"] == 0, scan["unreadable_samples"]
print(f"{scan['scanned_files']} files, {len(scan['patients'])} patients, "
      f"{scan['unreadable_files']} unreadable")
print("\nROI names across the cohort:")
print(", ".join(scan["roi_names"]))

In [ ]:
rows = []
for patient in scan["patients"]:
    for study in patient["studies"]:
        for s in study["series"]:
            rows.append({
                "patient": patient["patient_id"],
                "description": s["series_description"],
                "slices": s["instance_count"],
                "spacing_z": s["slice_spacing"],
                "uniform_z": s["slice_spacing_uniform"],
                "n_structs": len(s["linked_rtstructs"]),
                "n_doses": len(s["linked_rtdoses"]),
                "struct_rules": ",".join(sorted({r["match_rule"] for r in s["linked_rtstructs"]})),
            })

inventory = pd.DataFrame(rows)
print(f"{len(inventory)} image series across {inventory.patient.nunique()} patients")
print(f"median image series per patient: {inventory.groupby('patient').size().median():.0f}")
inventory.head(12)

In [ ]:
# How much of the cohort rests on a guess?
print("structure-set link rules:")
print(inventory.struct_rules.replace("", "(none)").value_counts().to_string())

guessed = inventory[inventory.struct_rules.str.contains("LargestSeriesFallback")]
print(f"\nseries whose structure link is a fallback guess: {len(guessed)}")

---
## 3 · Choose the right series

Here is the problem this collection poses. A single study holds:

- a **planning CT** — the full-length scan the plan was computed on;
- several **aligned CBCTs** — resampled onto the planning grid, so they report the same modality
  *and* the same frame of reference;
- a structure set per image series, named `BSPC_…` for the planning CT and `BSCB_…` for the CBCTs;
- one dose.

Nothing about `Modality` distinguishes them, and convert without choosing and you get every CBCT
alongside the planning CT.

The obvious heuristic — *keep the series with the most slices* (`--prefer-largest-series`) —
**does not work on this collection**, and it is worth seeing why before reaching for it. The
CBCTs were resampled onto the planning CT's grid, so they have the same spacing *and the same
slice count*. The rule ties, breaks the tie on SeriesInstanceUID, and picks arbitrarily. The
cell below shows the ties in your own download.

What does separate them is the **structure sets**, which are named for what they were drawn on:
`BSPC_…` on the planning CT, `BSCB_…` on the CBCTs. So select on that instead:

```
--struct-description BSPC
```

That keeps the image series carrying a matching structure set, and exports *that* set rather
than whichever happened to be linked first.

> **Why the dose still comes along.** An RT-DOSE names no image series — it points at an RT-PLAN,
> which this collection does not ship — so the only thing tying it to an image is the shared
> frame of reference, which every one of these series has. Rather than guess one owner (and
> orphan the dose whenever you select a different series), the scanner links the dose to every
> series in that frame. Selection then decides what gets exported, and the dose follows.

> **Same thing in the GUI:** the DICOM → NIfTI window shows the patient/study/series tree with
> slice counts and structure sets, and you tick what you want. The CLI selectors exist so an
> unattended run can make the same choice.

In [ ]:
# Does "most slices" actually identify the planning CT? Count how many series tie at each
# patient's maximum. A tie means the rule is choosing arbitrarily.
ties = (inventory.groupby("patient")
        .apply(lambda g: (g.slices == g.slices.max()).sum(), include_groups=False)
        .rename("series_tied_at_max"))

print(ties.to_string())
print(f"\npatients where the largest series is ambiguous: {(ties > 1).sum()} of {len(ties)}")

In [ ]:
# The structure sets, in contrast, say what they were drawn on.
struct_names = sorted({d for row in scan["patients"]
                       for st in row["studies"]
                       for s in st["series"]
                       for d in [r["series_description"] for r in s["linked_rtstructs"]]})
print("structure-set descriptions in this cohort:")
for n in struct_names:
    print("  ", n)

STRUCT_FILTER = "BSPC"
matched = inventory_rows = 0
for row in scan["patients"]:
    for st in row["studies"]:
        for s in st["series"]:
            inventory_rows += 1
            if any(STRUCT_FILTER.lower() in r["series_description"].lower()
                   for r in s["linked_rtstructs"]):
                matched += 1
print(f"\nseries matching --struct-description {STRUCT_FILTER}: {matched} "
      f"of {inventory_rows} (expect one per patient)")

---
## 4 · Survey the cohort

`--cohort-manifest` writes one CSV row per series: the three identifier columns, the voxel
spacing, and one column per ROI holding that structure's volume in cc. It is the cohort's index,
and it is what you read before deciding anything about the data.

Two honest notes:

- **Volumes cost a rasterization pass.** There is no analytic contour-area shortcut in this
  toolkit — a volume in cc means the mask was actually built. `--no-volumes` skips it and leaves
  every volume cell at `-1`.
- **Spacing is reported native here**, because `--output-spacing` is deliberately not passed.
  Surveying for geometry outliers only works if you are looking at the grid the scanner
  produced, not the grid you intend to resample onto.

> **Use the same `--salt` on every command that touches a cohort.** The manifest merges on its
> three identifier columns, so a survey run with real identifiers followed by a conversion run
> with hashed ones does not update those rows — it appends a second set, and the real
> identifiers stay behind in the cohort root next to the anonymized export. That is the whole
> thing anonymizing was meant to prevent, so the call below anonymizes too.

In [ ]:
manifest_run = run_cli(
    "--cohort-manifest",
    "--input", DICOM_DIR,
    "--output", NIFTI_DIR,
    "--struct-description", STRUCT_FILTER,
    "--require-structures",
    "--require-dose",
    # Same anonymization as the conversion in section 9, deliberately. The manifest merges on
    # the identifier columns, so surveying with real identifiers and then converting with
    # hashed ones would not update those rows — it would append a second set, leaving real
    # patient identifiers sitting in the cohort root next to an anonymized export.
    "--anonymize", "--salt", SALT,
    parse_json=True,
)

print(f"{manifest_run['series_count']} series surveyed")
print("ROI columns:", manifest_run["roi_columns"])
if manifest_run["skipped"]:
    print(f"\n{len(manifest_run['skipped'])} series skipped; reasons:")
    print(pd.Series([s["reason"] for s in manifest_run["skipped"]]).value_counts().to_string())

In [ ]:
manifest = pd.read_csv(manifest_run["manifest_csv"])

# Missing volumes are written as -1, not left blank, so a reader never has to guess whether an
# empty cell means "absent structure" or "truncated file". Convert to NaN before any statistics.
roi_cols = manifest_run["roi_columns"]
manifest[roi_cols] = manifest[roi_cols].replace(-1, np.nan)

print(manifest[["SpacingX", "SpacingY", "SpacingZ"]].describe().round(3).to_string())
manifest.head()

---
## 5 · Spot the outliers

The manifest is a quality-control instrument. Three things it surfaces:

1. **Odd slice spacing** — a series reconstructed differently from the rest of the cohort.
2. **ROI volumes far from the cohort norm** — often a mislabelled or partially contoured structure.
3. **Blank cells** — a structure that is simply missing for that patient.

The rule below is a plain interquartile-range flag. It is not clever, and it is not meant to
decide anything for you; it is meant to produce a short list worth opening in a viewer.

In [ ]:
def flag_outliers(values, k=1.5):
    """Return a boolean mask of points outside k interquartile ranges of the quartiles."""
    v = pd.Series(values).astype(float)
    q1, q3 = v.quantile(0.25), v.quantile(0.75)
    iqr = q3 - q1
    return (v < q1 - k * iqr) | (v > q3 + k * iqr)


flagged = pd.DataFrame({"PatientID": manifest.PatientID})
flagged["spacing_z"] = flag_outliers(manifest.SpacingZ)
for col in roi_cols:
    if manifest[col].notna().sum() >= 4:
        flagged[col] = flag_outliers(manifest[col])

flagged["n_flags"] = flagged.drop(columns=["PatientID"]).sum(axis=1)
review = flagged[flagged.n_flags > 0].sort_values("n_flags", ascending=False)

print(f"{len(review)} of {len(manifest)} series flagged for review")
review.head(10)

In [ ]:
present = [c for c in roi_cols if manifest[c].notna().sum() >= 4]
biggest = manifest[present].mean().idxmax() if present else None

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].hist(manifest.SpacingZ.dropna(), bins=20, color="#007DBA")
axes[0].set_xlabel("slice spacing (mm)"); axes[0].set_ylabel("series")
axes[0].set_title("Through-plane spacing")

if biggest:
    axes[1].hist(manifest[biggest].dropna(), bins=20, color="#1C7293")
    axes[1].set_xlabel(f"{biggest} volume (cc)")
    axes[1].set_title(f"{biggest} volume")
fig.tight_layout()
plt.show()

In [ ]:
# Non-uniform slice spacing deserves its own check. A series reconstructed with mixed gaps
# (3 mm through the target, 6 mm elsewhere) reads back a single averaged spacing from any tool
# that assumes a regular grid, which shifts contour planes during rasterization. The scan
# reports uniformity per series, so it can be caught here rather than in a trained model.
nonuniform = inventory[inventory.uniform_z == False]
print(f"series with non-uniform slice spacing: {len(nonuniform)}")
if len(nonuniform):
    print(nonuniform[["patient", "description", "slices", "spacing_z"]].to_string(index=False))

---
## 6 · Normalize the ROI names

Structure names are free text, and every institution writes them differently. Before a cohort is
usable, `Pancreas`, `PANCREAS` and `Pancreas_Ant` need to resolve to one label.

An **association** maps a canonical name to the aliases that should collapse into it. The file
below is the same format the desktop app imports and exports, so a mapping curated in the GUI
can be handed to a batch run unchanged.

Look at the ROI names printed in §2 before editing this — the aliases only help if they match
what is actually in your copy of the collection.

> **Same thing in the GUI:** Export Options → *Edit ROI Associations…*

In [ ]:
ASSOCIATIONS = [
    {"CanonicalName": "Pancreas",   "Aliases": ["pancreas", "Pancreas", "PANCREAS"]},
    {"CanonicalName": "Duodenum",   "Aliases": ["duodenum", "Duodenum", "DUODENUM"]},
    {"CanonicalName": "Stomach",    "Aliases": ["stomach", "Stomach", "STOMACH"]},
    {"CanonicalName": "SmallBowel", "Aliases": ["small_bowel", "SmallBowel", "Small Bowel", "Bowel_Small"]},
    {"CanonicalName": "Liver",      "Aliases": ["liver", "Liver", "LIVER"]},
    {"CanonicalName": "Kidney_L",   "Aliases": ["kidney_l", "Kidney_L", "LeftKidney", "Kidney-Left"]},
    {"CanonicalName": "Kidney_R",   "Aliases": ["kidney_r", "Kidney_R", "RightKidney", "Kidney-Right"]},
    {"CanonicalName": "SpinalCord", "Aliases": ["spinalcord", "SpinalCord", "Cord", "SpinalCanal"]},
]

ASSOC_PATH = WORK_DIR / "associations_pancreas.json"
ASSOC_PATH.write_text(json.dumps(ASSOCIATIONS, indent=2))

# Which of the cohort's actual ROI names does this cover?
covered = {a.lower() for entry in ASSOCIATIONS for a in entry["Aliases"]}
unmatched = [r for r in scan["roi_names"] if r.lower() not in covered]
print(f"{len(scan['roi_names']) - len(unmatched)} of {len(scan['roi_names'])} ROI names covered")
print("uncovered:", ", ".join(unmatched[:20]) or "(none)")

---
## 7 · Choose an output voxel size

Every series in a cohort should sit on the same grid, or nothing downstream can batch them.
Two defensible choices:

- **isotropic** (`1 × 1 × 1 mm`) — needed if you intend 3D convolutions to see the same physical
  extent along every axis;
- **native-like** (`1 × 1 × 3 mm`) — cheaper, and honest about the fact that the through-plane
  resolution was never 1 mm to begin with. That is what `OUTPUT_SPACING` uses here.

Resampling is not one operation. Interpolation must match what the voxels *mean*:

| volume | interpolation | why |
|---|---|---|
| image | linear | intensities are continuous |
| dose | linear | dose is a continuous field |
| masks | **nearest neighbour** | a label is categorical — interpolating produces voxels that are 0.4 of a structure, which is not a thing |

The toolkit applies this automatically; passing `--output-spacing` is the whole interface.

In [ ]:
print("output spacing:", OUTPUT_SPACING, "mm")
native = manifest[["SpacingX", "SpacingY", "SpacingZ"]].median().round(3).tolist()
print("cohort median native spacing:", native, "mm")

---
## 8 · Decide what metadata to carry

A NIfTI is a grid of numbers. Everything clinical — the patient's age, the scanner, the kVp, the
dose units — lives in the DICOM headers and is lost unless you deliberately carry it across.

The toolkit writes a `metadata.json` beside each exported series, in three sections matching the
three source objects. You choose the contents per section:

- `--metadata-tags` → `ImageAttributes`
- `--metadata-structure-tags` → `StructureAttributes`
- `--metadata-dose-tags` → `DoseAttributes`

Selections are **fo-dicom keywords** (`PatientAge`, `KVP`, `DoseUnits`) rather than
`group|element` strings. Alongside them are computed values, prefixed `@`, that are derived at
export time rather than copied from a tag — `@VoxelSize` reports the grid actually written,
which after resampling is *not* what any source header says. `@MaxDose` reads the dose volume
and applies `DoseGridScaling`, so it comes out in Gy rather than raw stored integers.

Computed values are scoped to their section: `@MaxDose` is meaningful under
`--metadata-dose-tags` and nowhere else, and asking for it elsewhere is rejected rather than
silently written as null.

> **Privacy.** Tags are copied verbatim. Never request `PatientName` or `PatientID` in an
> anonymized export — you would write the identifiers straight back into the file whose folder
> name you just hashed.

In [ ]:
IMAGE_TAGS     = ["PatientAge", "PatientSex", "Manufacturer", "ManufacturerModelName",
                  "KVP", "SliceThickness", "@VoxelSize", "@ImageDimensions"]
STRUCTURE_TAGS = ["StructureSetLabel", "@RoiNames", "@RoiCount"]
DOSE_TAGS      = ["DoseUnits", "DoseType", "DoseSummationType", "@MaxDose", "@DoseVoxelSize"]

for name, tags in [("image", IMAGE_TAGS), ("structures", STRUCTURE_TAGS), ("dose", DOSE_TAGS)]:
    print(f"{name:11} {', '.join(tags)}")

---
## 9 · Convert

One call does the whole cohort: image, one mask per ROI, every linked dose, the metadata
sidecar, and the manifest.

`--anonymize` replaces the identifiers in folder names, the manifest and the JSON with salted
hashes, and writes `AnonymizationKey.json` mapping them back. Because the hash is deterministic
in the salt, re-running with the same `SALT` puts the same patient in the same folder — which is
what makes §11 (growing the cohort) work.

> **`AnonymizationKey.json` is re-identification data.** Keep it out of version control and off
> shared drives. The example's `.gitignore` already excludes it.

In [ ]:
convert = run_cli(
    "--cohort-convert",
    "--input", DICOM_DIR,
    "--output", NIFTI_DIR,
    "--struct-description", STRUCT_FILTER,
    "--require-structures",
    "--require-dose",
    "--associations", ASSOC_PATH,
    "--output-spacing", ",".join(str(v) for v in OUTPUT_SPACING),
    "--anonymize", "--salt", SALT,
    "--metadata-tags", ",".join(IMAGE_TAGS),
    "--metadata-structure-tags", ",".join(STRUCTURE_TAGS),
    "--metadata-dose-tags", ",".join(DOSE_TAGS),
    parse_json=True,
)

print(f"succeeded: {convert['succeeded']}   failed: {convert['failed']}")
for err in convert["errors"][:5]:
    print("  !", err["patient_id"], err["message"])

In [ ]:
def show_tree(root, max_entries=4):
    """Print the first few exported cases so the layout is visible."""
    root = Path(root)
    for i, meta in enumerate(sorted(root.rglob("metadata.json"))):
        if i >= max_entries:
            print("  ...")
            break
        case = meta.parent
        print(case.relative_to(root).as_posix())
        for f in sorted(case.rglob("*.nii.gz")):
            print("   ", f.relative_to(case).as_posix())


show_tree(convert["output_root"])

---
## 10 · Verify

Three things worth checking before trusting an exported cohort:

1. **image and masks share a grid** — if they do not, every overlay and every loss function
   downstream is quietly misaligned;
2. **the metadata sidecar carries what you asked for**, dose section included;
3. **the masks look right** — no substitute for putting eyes on a slice.

> **Dose is the exception, deliberately.** Masks are rasterized onto the image grid, so they come
> out voxel-aligned with it. The dose is not: it is resampled to the same *spacing* you asked for,
> but it keeps its own origin and extent, because a dose grid typically covers only the region
> around the target and re-gridding it onto the full image would inflate it with zeros. So
> `dose.GetSize() != image.GetSize()` is expected, not a defect — but it does mean **you must
> resample before overlaying or indexing the two together**, which the figure below does.

In [ ]:
case = convert["series"][0]
root = Path(convert["output_root"])

image = sitk.ReadImage(str(root / case["image"]))
print("image ", image.GetSize(), [round(s, 3) for s in image.GetSpacing()])

for mask in case["masks"]:
    m = sitk.ReadImage(str(root / mask["file"]))
    assert m.GetSize() == image.GetSize(), f"{mask['name']} grid differs from the image"
    assert np.allclose(m.GetSpacing(), image.GetSpacing())
    print(f"mask   {mask['name']:12} {mask['volume_cc']:8.2f} cc  [same grid]")

for dose_path in case["doses"]:
    d = sitk.ReadImage(str(root / dose_path))
    print("dose  ", d.GetSize(), [round(s, 3) for s in d.GetSpacing()],
          "(own extent — resample onto the image before combining)")
    assert np.allclose(d.GetSpacing(), image.GetSpacing()), "dose spacing does not match the export grid"

In [ ]:
meta = json.loads((root / case["output_dir"] / "metadata.json").read_text())
for section in ("ImageAttributes", "StructureAttributes", "DoseAttributes"):
    print(f"--- {section} ---")
    print(json.dumps(meta.get(section, {}), indent=2)[:600])

In [ ]:
# Overlay the largest structure and the dose on the slice where that structure is biggest.
target = max(case["masks"], key=lambda m: m["volume_cc"])
mask_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(root / target["file"])))
img_arr = sitk.GetArrayFromImage(image)
z = int(np.argmax(mask_arr.sum(axis=(1, 2))))

fig, axes = plt.subplots(1, 2, figsize=(11, 5.2))
for ax in axes:
    ax.imshow(img_arr[z], cmap="gray")
    ax.axis("off")

axes[0].imshow(np.ma.masked_where(mask_arr[z] == 0, mask_arr[z]), cmap="autumn", alpha=0.45)
axes[0].set_title(f"{target['name']} — slice {z} ({target['volume_cc']:.1f} cc)")

if case["doses"]:
    # Resample the dose ONTO the image grid first. The two share a coordinate system but not an
    # extent, so indexing dose[z] against image[z] without this would silently plot the wrong
    # part of the patient. Linear interpolation, and 0 Gy outside the dose grid.
    dose_img = sitk.ReadImage(str(root / case["doses"][0]))
    dose_on_image = sitk.Resample(
        dose_img, image, sitk.Transform(), sitk.sitkLinear, 0.0, dose_img.GetPixelID())

    dose_arr = sitk.GetArrayFromImage(dose_on_image)
    washed = np.ma.masked_where(dose_arr[z] < 0.1 * dose_arr.max(), dose_arr[z])
    im = axes[1].imshow(washed, cmap="jet", alpha=0.55)
    fig.colorbar(im, ax=axes[1], fraction=0.046, label="dose (Gy)")
    axes[1].set_title(f"dose — slice {z} (max {dose_arr.max():.1f} Gy)")
else:
    axes[1].set_title("no dose for this case")

fig.tight_layout()
plt.show()

---
## 11 · Grow the cohort

Real datasets accumulate. Two properties make that safe:

- **the anonymization hash is deterministic in the salt**, so the same patient always lands in
  the same folder — no duplicate-under-a-new-name;
- **the manifest merges rather than regenerates**, keyed on the three identifier columns. An
  existing row is updated in place, a new one is appended, and a new ROI column is added without
  disturbing the columns already there.

So re-running the conversion after pulling more patients extends the dataset. The cell below
re-runs it unchanged and confirms nothing moved.

In [ ]:
before = pd.read_csv(convert["output_root"] + "/" + convert["manifest_csv"])
first_key = before.iloc[0][["PatientID", "StudyUID", "SeriesUID"]].tolist()

again = run_cli(
    "--cohort-convert",
    "--input", DICOM_DIR, "--output", NIFTI_DIR,
    "--struct-description", STRUCT_FILTER, "--require-dose",
    "--associations", ASSOC_PATH,
    "--output-spacing", ",".join(str(v) for v in OUTPUT_SPACING),
    "--anonymize", "--salt", SALT,
    "--metadata-tags", ",".join(IMAGE_TAGS),
    "--metadata-structure-tags", ",".join(STRUCTURE_TAGS),
    "--metadata-dose-tags", ",".join(DOSE_TAGS),
    parse_json=True,
)

after = pd.read_csv(again["output_root"] + "/" + again["manifest_csv"])
assert after.iloc[0][["PatientID", "StudyUID", "SeriesUID"]].tolist() == first_key, \
    "identifiers moved between runs — the salt is not being applied consistently"
assert len(after) == len(before), "re-running the same cohort duplicated rows"
print(f"stable: {len(after)} rows, same hashes, no duplicates")

---
## 12 · Back to DICOM

Everything so far has been the forward direction, and it is where most tooling stops. But a
model that produces masks has not helped anyone until those masks are back in a structure set a
treatment planning system will open. That is the last mile, and it is the direction this toolkit
exists for.

`--reverse` takes a folder of per-ROI masks and writes an RT structure set. With a reference
DICOM series it reuses that series' UIDs so the result overlays the original study. Without one
it synthesizes the geometry from `metadata.json` — useful when all you have is NIfTI.

The round trip is lossy in one direction that is worth understanding: contours become a binary
mask, and a binary mask becomes contours again. Whatever the mask could not represent is already
gone by the time the writer runs. §13 quantifies that loss against known-answer geometry;
here we just measure it on real anatomy.

In [ ]:
case_dir = root / case["output_dir"]
regen_rtstruct = WORK_DIR / "roundtrip" / "rtstruct_regenerated.dcm"
regen_rtstruct.parent.mkdir(parents=True, exist_ok=True)

# No --metadata here, deliberately. The metadata.json the forward direction wrote in §9 is the
# DICOM-*tag* sidecar (ImageAttributes / StructureAttributes / DoseAttributes); the reverse
# direction's metadata.json is an unrelated schema carrying patient, study, frame of reference
# and rescale slope. Passing the tag sidecar to --metadata is silently ignored, so it buys
# nothing and reads as if provenance were being carried. Nothing the forward direction writes
# can drive the reverse one: to make a regenerated structure set land on the ORIGINAL study,
# point --image-folder at the source DICOM series instead. Here the study is synthesized with
# fresh anonymous UIDs, which is exactly right for measuring what the round trip costs.
#
# Note this run also writes a reverse-schema metadata.json INTO case_dir/masks/, so that a
# repeat run reuses the same UIDs rather than minting a new series. It is a write into an
# input folder, and it is not the sidecar read in §10.
run_cli(
    "--reverse",
    "--masks-folder", case_dir / "masks",
    "--image-nifti", case_dir / "image.nii.gz",
    "--output", regen_rtstruct,
    "--output-image-folder", WORK_DIR / "roundtrip" / "dicom",
)
print("wrote", regen_rtstruct.name, f"({regen_rtstruct.stat().st_size/1024:.0f} KB)")

In [ ]:
# Convert the regenerated structure set straight back to masks, on the same grid, and compare.
roundtrip_dir = WORK_DIR / "roundtrip" / "masks"
run_cli(
    "--forward",
    "--rtstruct", regen_rtstruct,
    "--image-folder", WORK_DIR / "roundtrip" / "dicom",
    "--output-folder", roundtrip_dir,
)


def dice(a, b):
    a, b = a.astype(bool), b.astype(bool)
    denom = a.sum() + b.sum()
    return 1.0 if denom == 0 else 2.0 * (a & b).sum() / denom


comparison = []
for mask in case["masks"]:
    rt = roundtrip_dir / Path(mask["file"]).name
    if not rt.exists():
        comparison.append({"roi": mask["name"], "dice": np.nan, "note": "not regenerated"})
        continue
    original = sitk.GetArrayFromImage(sitk.ReadImage(str(root / mask["file"])))
    returned = sitk.GetArrayFromImage(sitk.ReadImage(str(rt)))
    vol0 = original.sum() * np.prod(image.GetSpacing()) / 1000.0
    vol1 = returned.sum() * np.prod(image.GetSpacing()) / 1000.0
    comparison.append({
        "roi": mask["name"],
        "dice": dice(original, returned),
        "cc_before": round(vol0, 2),
        "cc_after": round(vol1, 2),
        "cc_delta_pct": round(100 * (vol1 - vol0) / vol0, 2) if vol0 else np.nan,
    })

roundtrip = pd.DataFrame(comparison)
print(roundtrip.to_string(index=False))

On this cohort those numbers come back at **Dice 1.000 and 0.00% volume change**, and that is
worth understanding rather than just accepting.

**The mask → contour → mask leg is lossless by construction when the two conventions agree.**
The structure-set writer traces contours along voxel *boundaries*; the rasterizer fills voxels
whose *centres* fall inside a contour. Those are the same set, so the second pass reproduces the
first exactly. A round-trip Dice below 1.0 here would indicate the two halves of this toolkit
disagree about where a boundary is — which is precisely the disagreement the benchmark found
*between* tools.

**So this measures self-consistency, not accuracy.** The information actually lost was lost
earlier, on the very first rasterization in §9, when smooth clinical contours became a voxel
grid — and comparing the toolkit to itself cannot see that loss. Only §13, against shapes whose
true volume is known in closed form, can.

Two caveats worth keeping:

- **Small structures are harder.** A structure only a few voxels across is nearly all boundary,
  and the exactness above depends on the traced polygon and the fill agreeing everywhere along
  it. Check your own smallest ROI rather than assuming the lungs generalize to a cord.
- **Some geometry cannot survive at any resolution.** An open, non-planar applicator track has no
  binary-mask representation and comes back a filled polygon. That is a property of the mask, not
  of this implementation, and it applies to every tool that round-trips through one.

---
## 13 · Prove it against a known answer

Everything above compares the toolkit to itself. That shows consistency, not correctness — a
rasterizer with a half-voxel bias round-trips beautifully and is still wrong.

The `rtmask-conformance` package settles it. It generates a synthetic CT and structure set whose
shapes are analytic — spheres, cubes, cylinders, a torus, a hollow sphere — so the true volume
and surface of each is known in closed form rather than estimated. It converts them with your
tool, then scores Dice, 95th-percentile Hausdorff distance, mean surface distance and volume
error against that ground truth.

This is the same gate the project runs in CI on Windows, Linux and macOS, so a failure here is a
real regression, not a notebook artifact.

In [ ]:
# Installs the tip of main. CI instead pins a SHA (see RTMASK_CONFORMANCE_REF in
# .github/workflows/conformance-crossplatform.yml) so the gate is reproducible -- append
# "@<sha>" below if you need this section to be reproducible too.
%pip install -q "rtmask-conformance @ git+https://github.com/brianmanderson/RTMaskConformanceTest"

In [ ]:
fixture = WORK_DIR / "conformance" / "fixture"
predictions = WORK_DIR / "conformance" / "predictions"

# Generating the analytic ground truth is the slow step here -- expect several minutes, because
# every primitive's partial-volume truth is integrated by quadrature. CI trades a little of that
# precision for speed with `--n-quadrature 2`; the defaults are used below.
subprocess.run([sys.executable, "-m", "rtmask_conformance.cli", "generate", str(fixture)],
               check=True, capture_output=True, text=True)

run_cli(
    "--forward",
    "--rtstruct", fixture / "rtstruct" / "primitives_planar.dcm",
    # The generated reference CT series lands in `refct/`, not `image/`.
    "--image-folder", fixture / "refct",
    "--output-folder", predictions,
)
print("primitives converted:", len(list(predictions.glob("*.nii.gz"))))

In [ ]:
verify = [sys.executable, "-m", "rtmask_conformance.cli", "verify",
          "--predictions", str(predictions),
          "--groundtruth", str(fixture / "groundtruth")]

# The repository's conformance.yaml carries documented per-primitive threshold adjustments.
# It is absent when this notebook runs outside a checkout, and the package defaults apply.
config = Path("../conformance.yaml")
if config.exists():
    verify += ["--config", str(config)]
    print("using", config.resolve())

result = subprocess.run(verify, capture_output=True, text=True)
print(result.stdout[-3000:])
assert result.returncode == 0, result.stdout + result.stderr
print("\nCONFORMANCE PASSED")

---
## Where this goes next

You now have a resampled, anonymized, dose-carrying NIfTI cohort with a growable manifest, a
demonstrated path back to DICOM, and a measured accuracy claim against closed-form truth.

- **Train on it.** Session 2 of the AAPM track builds a PyTorch segmentation pipeline on a
  dataset in exactly this layout.
- **Compare tools.** `PythonCode/README.md` in the parent repository reproduces the six-tool
  benchmark that measures how much rasterizer choice actually changes a mask.
- **Use the GUI.** Everything here has a point-and-click equivalent — see
  [`GUI_WALKTHROUGH.md`](GUI_WALKTHROUGH.md).

### Cleaning up

The working directory holds the raw DICOM, the exported cohort, and the downloaded binary.
Delete `WORK_DIR` when you are done — and remember that `AnonymizationKey.json` is
re-identification data, so it should not outlive your need for it.